# Pritam — Day 2 Session Summary
**Sprint:** Dark Store + Integrated Logistics | **Date:** April 2, 2026  
**Branch:** `pritam_temp_apr1` | **Role:** Lead Coder + Integration Architect

---

This notebook is a chronological account of everything done in this session:  
git operations, code written, bugs fixed, PR review addressed, and outputs produced.

Load [`session_summary_v3.md`](../../session_summary_v3.md) for full project context  
Load [`docs/pritam_sess_summ.md`](../../docs/pritam_sess_summ.md) for Pritam-scoped context

---
## 1. Git Setup — Branch & Sync

### What was done
1. **Pulled `origin/main`** — 6 new commits fetched (team Day 1 work merged in):  
   - `01_data_pipeline.ipynb`, `01_exploratory_analysis.ipynb`  
   - `src/exploratory_analysis.py`  
   - Two Folium HTML visualisations (`sp_eda_map.html`, `olist_er_diagram.html`)
2. **Created and switched to `pritam_temp_apr1`** — all Day 2 work done here.

### Commands used
```bash
git pull origin main
git checkout -b pritam_temp_apr1
```

### Why a feature branch?
The roadmap risk register flags git merge conflicts as HIGH likelihood.  
Each person works in their own branch and Pritam resolves conflicts daily before merging to `main`.

---
## 2. Data Acquisition — `master_df.parquet`

### Dependency
Per the roadmap, Pritam's Day 2 task (distance matrix) is **blocked** until Vybhav commits `master_df.parquet`.  
`*.parquet` is gitignored (large binary), so it must be transferred out-of-band.

### What was done
- Verified `master_df.parquet` was available on the team Google Drive  
- Downloaded from: `https://drive.google.com/drive/folders/1l7DQfNBa6IPuf6s_ytjAAgR9FsJ8tBDn`  
- Placed at: `data/master_df.parquet` (7.7 MB)

### File summary
| Property | Value |
|----------|-------|
| Rows | 41,731 (all SP state) |
| Columns | 33 |
| Key columns | `customer_lat`, `customer_lon`, `customer_zip_code_prefix`, `customer_state` |
| Source | Vybhav's `src/data_pipeline.py` output |

### Note on gitignore
```
data/*.parquet    # in .gitignore — not committed, downloaded manually
data/*.csv        # same — all data artefacts are regenerated on each run
*.npy / *.pkl     # same
```

---
## 3. Core Day 2 Deliverable — `src/haversine_matrix.py`

### Objective
Build a **500×500 pairwise Haversine distance matrix** in integer-scaled km×1000 format,  
ready to feed directly into OR-Tools as arc costs on Day 3.

### What was done
Rewrote the Day 1 stub in `src/haversine_matrix.py` into a full production module.  
Five public functions plus a `run()` pipeline entry point.

---
### 3.1 `stratified_sample(df, n=500, random_state=42)`

**Problem:** A naive `df.sample(500)` would over-represent high-volume city-centre zips,  
producing a matrix that only describes short intra-city distances.

**Solution — proportional stratification by zip code:**

```
1. Drop rows with null lat/lon
2. Compute zip_counts from the full (non-deduplicated) base  →  true order volume weight
3. Allocate 500 sample slots proportionally: quota_i = round(count_i / total * n)
4. Deduplicate on exact (lat, lon) to form the candidate pool per zip
5. Draw min(quota, available) unique rows per zip
6. Post-check: top-up any shortfall from the global remaining pool
```

**Key design decision:** `zip_counts` computed from **non-deduplicated** data.  
A high-density zip with 1,000 orders but 5 distinct geocodes should still get proportionally more slots than a sparse zip with 50 orders.

**Remainder handling (rounding edge cases):**
- Positive remainder → largest fractional-part zips each get +1 slot  
- Negative remainder → largest-quota zips each lose 1 slot  
- Under-delivery → global pool top-up fills any shortfall

**Output columns:** `node_id, customer_lat, customer_lon, customer_zip_code_prefix, order_count`

---
### 3.2 `build_distance_matrix(coords)` — vectorised Haversine

**The Haversine formula** computes great-circle distance between two points:

$$a = \sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos\phi_1 \cdot \cos\phi_2 \cdot \sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$

$$d = 2R \cdot \arcsin\!\left(\sqrt{a}\right), \quad R = 6{,}371 \text{ km}$$

**Vectorised via NumPy broadcasting (not a loop):**
```python
dlat = lat[:, None] - lat[None, :]    # (500,1) - (1,500) → (500,500)
dlon = lon[:, None] - lon[None, :]    # same
a    = sin(dlat/2)**2 + cos(lat[:,None]) * cos(lat[None,:]) * sin(dlon/2)**2
d    = 2R * arcsin(sqrt(clip(a, 0, 1)))
```
**Speed:** ~10 ms for 500×500 vs ~8 s for a Python double loop (800× faster)

**Integer scaling:** `np.rint(d * 1000).astype(np.int64)` — rounds (not truncates).  
A plain `.astype(int64)` truncates: 6.999 km → `6999` instead of `7000`.  
`np.rint()` eliminates the systematic downward bias across 250,000 off-diagonal entries.

---
### 3.3 `validate_matrix(matrix)` → `dict`

Five checks — all raise `ValueError` (not `AssertionError`) so they fire under `python -O`:

| Check | Why |
|-------|-----|
| Square 2D | OR-Tools requires NxN |
| `dtype == int64` | OR-Tools requires integer arc costs |
| Symmetric | Haversine is symmetric by construction |
| Diagonal == 0 | Self-distance must be zero |
| All off-diagonal > 0 | No zero arc costs — prevents degenerate routes |

Returns: `{'shape', 'dtype', 'min_km', 'mean_km', 'max_km', 'is_symmetric', 'diagonal_zero', 'all_positive_off_diag'}`

---
### 3.4 `save_distance_matrix()` / `load_distance_matrix()`

Binary `.npy` round-trip — preserves `int64` dtype exactly.  
Pranav's `vrp_nodes.py` and Pritam's OR-Tools CVRPTW both call `load_distance_matrix()` on Day 3.

---
### 3.5 `run()` — one-shot pipeline entry point

```python
sample_df, matrix, stats = run(
    parquet_path    = "data/master_df.parquet",
    matrix_path     = "data/distance_matrix.npy",
    sample_csv_path = "data/sp_customer_sample.csv",
    n               = 500,
    random_state    = 42,
)
```

5-step logged pipeline — each step prints progress to stdout.  
Reproducible: same `random_state` always produces identical outputs.

---
## 4. Outputs Generated

| File | Size | Detail |
|------|------|--------|
| `data/distance_matrix.npy` | 1,953 KB | 500×500 int64, km×1000 |
| `data/sp_customer_sample.csv` | ~30 KB | 500 rows, 5 cols |
| `outputs/day2_geo_map.html` | ~2.4 MB | Folium interactive geo map |

### Matrix validation stats

| Metric | Value | Note |
|--------|-------|------|
| Shape | (500, 500) | ✅ |
| dtype | int64 | ✅ OR-Tools compatible |
| min | 0.048 km | ✅ |
| mean | 146.24 km | ✅ SP state extent |
| max | 745.54 km | ✅ SP state ~900 km E–W |
| Symmetric | True | ✅ |
| Diagonal zero | True | ✅ |
| All off-diag > 0 | True | ✅ No zero arc costs |

> **Note on distance range:** The sample covers the full SP *state* bounding box.  
> For single dark-store zone VRP (Day 3), Pranav's `vrp_nodes.csv` will subset  
> to city-level customers, so effective arc distances will be in the 0.5–25 km range.

In [1]:
# Quick verification — reload outputs and confirm stats
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, "../..")

from src.haversine_matrix import load_distance_matrix, validate_matrix

matrix = load_distance_matrix("../../data/distance_matrix.npy")
stats  = validate_matrix(matrix)
sample = pd.read_csv("../../data/sp_customer_sample.csv")

print("=== distance_matrix.npy ===")
print(f"  shape  : {stats['shape']}")
print(f"  dtype  : {stats['dtype']}")
print(f"  min_km : {stats['min_km']:.3f}")
print(f"  mean_km: {stats['mean_km']:.2f}")
print(f"  max_km : {stats['max_km']:.3f}")
print(f"  symmetric          : {stats['is_symmetric']} ✓")
print(f"  diagonal_zero      : {stats['diagonal_zero']} ✓")
print(f"  all_positive_offdiag: {stats['all_positive_off_diag']} ✓")
print()
print("=== sp_customer_sample.csv ===")
print(f"  rows: {len(sample)}  |  cols: {list(sample.columns)}")
sample.head(3)

=== distance_matrix.npy ===
  shape  : (500, 500)
  dtype  : int64
  min_km : 0.048
  mean_km: 146.24
  max_km : 745.537
  symmetric          : True ✓
  diagonal_zero      : True ✓
  all_positive_offdiag: True ✓

=== sp_customer_sample.csv ===
  rows: 500  |  cols: ['node_id', 'customer_lat', 'customer_lon', 'customer_zip_code_prefix', 'order_count']


,node_id,customer_lat,customer_lon,customer_zip_code_prefix,order_count
0,0,-23.546081,-46.643810,1046,18
1,1,-23.545495,-46.650239,1222,18
2,2,-23.541701,-46.652571,1224,23


---
## 5. PR #4 — Review & Fixes

PR #4 was raised: *"Day 2 (Pritam): Vectorised Haversine Distance Matrix + Day 2 Summary Notebook"*  
**1 Copilot automated review (requesting changes) + 3 Vybhav comments.**

### Copilot review — 8 comments addressed

| # | File | Issue | Fix |
|---|------|-------|-----|
| C1 | `haversine_matrix.py` | `total` summed mapped column, not raw counts | `total = zip_counts.sum()` |
| C2 | `haversine_matrix.py` | `zip_counts` computed after dedup → distorted weights | Compute from non-deduped `base`; dedup only candidate pool |
| C3 | `haversine_matrix.py` | `clip(lower=1)` could make remainder negative | Added `elif remainder < 0` block reducing from largest-quota zips |
| C4 | `haversine_matrix.py` | Under-sampling not handled | Global pool top-up via merge+anti-join |
| C5 | `haversine_matrix.py` | Docstring range ≠ actual stats | Updated to min≈0.05 / mean≈146 / max≈746 km (SP state) |
| C6 | `haversine_matrix.py` | `.astype(int64)` truncates, not rounds | `np.rint()` before cast |
| C7 | `haversine_matrix.py` | `assert` skipped under `python -O` | Replaced with `raise ValueError` |
| C8 | `pyproject.toml` | Tabs in dependency array | Converted to spaces |

### Vybhav review — 3 comments addressed

| # | Comment | Fix |
|---|---------|-----|
| V1 | Remove 
| V2 | (minor inline style) | Fixed |
| V3 | Move session summary to `docs/` | `pritam_sess_summ.md` → `docs/pritam_sess_summ.md` |

All fixes committed as `54b66b4` and pushed to the open PR — no new PR needed.

---
## 6. Day 2 Summary Notebook — `notebooks/02_distance_matrix.ipynb`

Created and updated through the session. **7 sections:**

| Section | Content |
|---------|--------|
| 1 | Data loading + SP filter |
| 2 | Stratified sampling theory + implementation |
| 3 | Haversine formula derivation + vectorisation + `np.rint()` explanation |
| 4 | Matrix build, validate, save |
| 5 | Visualisations (matplotlib: distribution, heatmap, top-25 distances) |
| 6 | **Folium geo visualisation** (HeatMap + MarkerCluster + bounding box) |
| 7 | OR-Tools 5-node toy VRP demo using real matrix + full API reference |

### Folium geo map (Section 6)

Interactive HTML map exported to `outputs/day2_geo_map.html`:

| Layer | Detail |
|-------|--------|
| HeatMap | All 41,731 SP customer rows — full demand density |
| MarkerCluster | 500 sampled nodes, blue→red by `order_count` |
| Unclustered | Same 500 nodes, togglable (useful at high zoom) |
| Bounding box | Purple dashed rectangle — SP state sample extent |
| Colour legend | Linear colormap bar |
| Layer control | Toggle each layer independently |

### Notebook rename
The linting PR from the team renamed:  
`02_day2_distance_matrix.ipynb` → `02_distance_matrix.ipynb`  
This was a clean auto-merge — no conflicts.

---
## 7. Merging Team Changes (Mid-Session Sync)

During the session, the team pushed new work to `origin/main`. Two syncs performed:

### Sync 1 — start of session (`git pull origin main`)
Pulled 6 commits — Vybhav's data pipeline, Anurag's EDA, visualisation files.

### Sync 2 — mid-session (`git merge origin/main`)
Merged commit `667e90f` — linting/styling pass across the repo:

| File changed | Change |
|-------------|--------|
| `notebooks/00_summary.ipynb` | Renamed from `00_day1_summary.ipynb` |
| `notebooks/02_distance_matrix.ipynb` | Renamed from `02_day2_distance_matrix.ipynb` |
| `src/clustering.py` | Lint fix |
| `src/data_pipeline.py` | Lint fix |
| `src/exploratory_analysis.py` | Lint fix |
| `src/demand_baseline.py` | **New file** — Vybhav's demand + baseline KPIs |
| `notebooks/02_demand_and_baseline.ipynb` | **New notebook** |
| `data/baseline_kpis.json` | **New file** — baseline KPI outputs |
| `visualisations/sp_eda_map.html` | Updated EDA map |
| `pyproject.toml` | Dependency updates |

Both merges were **clean auto-merges** — zero conflicts.  
This confirms the team's "separate files/folders per person" strategy is working.

---
## 8. Git Commit Log — This Session

```
24ad971  Merge remote-tracking branch 'origin/main' into pritam_temp_apr1
5ac80d7  Day 2 notebook: add Folium geo visualisation (HeatMap + MarkerCluster + bbox)
667e90f  (origin/main) Linting/Styling checks across the repo           ← team commit
0641728  Merge pull request #4 from metaphorpritam/pritam_temp_apr1     ← PR #4 merged
54b66b4  Address PR #4 review comments (Copilot x8 + Vybhav x3)
c7875ba  Day 2 (Pritam): update session summary + notebook polish + dependency sync
```

**Branch:** `pritam_temp_apr1` — pushed to `origin/pritam_temp_apr1`  
**PR #4 status:** Merged ✅

---
## 9. Day 2 EOD Checklist

Per the roadmap (page 6), Day 2 EOD checkpoint for Pritam:

| Checkpoint | Status | Detail |
|-----------|--------|--------|
| `distance_matrix.npy` saved (500×500 Haversine, integer-scaled) | ✅ | 1,953 KB, int64 |
| Validate: symmetric | ✅ | `matrix == matrix.T` |
| Validate: diagonal zero | ✅ | |
| Validate: all off-diag > 0 | ✅ | No zero arc costs |
| `sp_customer_sample.csv` saved (500 points) | ✅ | 500 rows, 5 cols |
| PR raised and review addressed | ✅ | PR #4 merged |
| `docs/pritam_sess_summ.md` updated | ✅ | Day 2 completion logged |

## 10. Day 3 Handoff Contract

What Pranav needs from today's outputs to build `vrp_nodes.csv`:

> Row `i` of `distance_matrix.npy` corresponds to `node_id == i` in `sp_customer_sample.csv`.  
> The depot must be prepended as `node_id = 0` — all customer node IDs shift by +1.  
> Arc cost between nodes `i` and `j` = `matrix[i][j]` (km × 1000, integer).

What Pritam needs on Day 3 to start OR-Tools CVRPTW:

| Dependency | Owner | ETA |
|-----------|-------|-----|
| `data/vrp_nodes.csv` | Pranav | Day 2 EOD |
| `data/dark_store_candidates.csv` | Sneha | Day 2 mid |
| `data/distance_matrix.npy` | Pritam (✅ done) | — |